# ACLED API Explorer (Bearer Auth)

This notebook uses bearer-token authentication for ACLED reads.
Run `acled_oauth_token.py` first to populate `ACLED_BEARER_TOKEN` in root `.env`.

In [ ]:
from urllib.parse import urlencode

SOURCE_NAME = "ACLED"
DATA_ENDPOINT = "https://acleddata.com/api/acled/read"
TOKEN_ENDPOINT = "https://acleddata.com/oauth/token"
CAPABILITIES = [
    "Event-level conflict data with geocoded locations.",
    "Filtering by date range, country, region, actor, and event type.",
    "Near-real-time updates with standardized event taxonomy.",
    "Structured fields useful for panel data and count aggregation.",
]
BEARER_QUERY_PARAMS = {
    "country": "Ukraine",
    "event_date": "2025-01-01|2025-01-31",
    "limit": "5",
}

print(SOURCE_NAME)
print("Auth method: Bearer token")
print("Token endpoint:")
print(TOKEN_ENDPOINT)
print("Data endpoint:")
print(DATA_ENDPOINT)
print("\nBearer request URL preview:")
print(f"{DATA_ENDPOINT}?{urlencode(BEARER_QUERY_PARAMS)}")
print("\nCapabilities:")
for item in CAPABILITIES:
    print("-", item)

In [ ]:
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json
import os
import time


def normalize_env_value(raw_value: str) -> str:
    if (
        len(raw_value) >= 2
        and raw_value[0] == raw_value[-1]
        and raw_value[0] in {'"', "'"}
    ):
        return raw_value[1:-1]
    return raw_value


def find_project_root(start_dir: Path) -> Path | None:
    for directory in [start_dir, *start_dir.parents]:
        if (directory / ".env").exists():
            return directory
    return None


def load_root_env() -> Path | None:
    project_root = find_project_root(Path.cwd())
    if project_root is None:
        return None
    env_path = project_root / ".env"
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), normalize_env_value(value.strip()))
    return project_root


def mask_secret(secret: str) -> str:
    if len(secret) <= 10:
        return "*" * len(secret)
    return f"{secret[:6]}...{secret[-4:]}"


project_root = load_root_env()

data_endpoint = globals().get("DATA_ENDPOINT", "https://acleddata.com/api/acled/read")
bearer_query_params = globals().get(
    "BEARER_QUERY_PARAMS",
    {"country": "Ukraine", "event_date": "2025-01-01|2025-01-31", "limit": "5"},
)

token = normalize_env_value(os.getenv("ACLED_BEARER_TOKEN", "").strip())
token_type = (
    normalize_env_value(os.getenv("ACLED_BEARER_TOKEN_TYPE", "Bearer").strip())
    or "Bearer"
)

if not token:
    print("Missing ACLED_BEARER_TOKEN in .env")
    print("Run: uv run python 'API explorer/acled/acled_oauth_token.py'")
else:
    request_url = f"{data_endpoint}?{urlencode(bearer_query_params)}"
    print("Using bearer token:")
    print(f"{token_type} {mask_secret(token)}")
    print("Live request URL:")
    print(request_url)

    payload = None
    last_error = None
    for attempt_index, wait_seconds in enumerate([0, 4, 8], start=1):
        if wait_seconds > 0:
            print(f"Waiting {wait_seconds}s before retry...")
            time.sleep(wait_seconds)
        try:
            request = Request(request_url, method="GET")
            request.add_header("Authorization", f"{token_type} {token}")
            request.add_header("Accept", "application/json")
            request.add_header(
                "User-Agent",
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            )
            request.add_header("Accept-Language", "en-US,en;q=0.9")
            request.add_header("Referer", "https://acleddata.com/")
            with urlopen(request, timeout=30) as response:
                payload = json.loads(response.read().decode("utf-8"))
            print(f"Bearer request succeeded on attempt {attempt_index}.")
            break
        except HTTPError as error:
            last_error = error
            print(f"Attempt {attempt_index} failed with HTTP {error.code}.")
            try:
                body = error.read().decode("utf-8", errors="replace").strip()
            except Exception:
                body = ""
            if body:
                print(f"Response body: {body}")
            if error.code == 403 and "error 1010" in body.lower():
                print("Detected Cloudflare 1010 block from ACLED edge.")
                print(
                    "Try from your normal browser network (disable VPN/proxy), then retry."
                )
            if error.code != 429:
                break
        except Exception as error:
            last_error = error
            print(f"Attempt {attempt_index} failed: {error}")
            break

    if payload is None:
        print("No live payload returned.")
        print(f"Last error: {last_error}")
    else:
        records = (
            payload.get("data") or payload.get("results") or payload.get("events") or []
        )
        print(f"\nRecords returned: {len(records)}")
        for index, record in enumerate(records[:5], start=1):
            event_date = record.get("event_date", "<missing event_date>")
            country = record.get("country", "<missing country>")
            event_type = record.get("event_type", "<missing event_type>")
            print(f"{index}. {event_date} | {country} | {event_type}")

        if project_root is None:
            output_dir = Path("outputs")
        else:
            output_dir = project_root / "API explorer" / "acled" / "outputs"
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path = output_dir / "acled_bearer_notebook_response.json"
        output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        print(f"\nSaved payload to: {output_path.resolve()}")